# M5 — Section-Routed Specialist Loss

Directly encodes Phase 1 findings:
- **Decision tokens** → CW-CE (E3's accuracy strength) + entropy matching (E5a's safety strength)
- **Explanation tokens** → NO supervision (E7's reasoning emergence)

The model must learn to generate good explanations *internally* to reach the correct decision.

### Fixes applied (2025-03-29)
- `memory_fraction` set to 0.85 (was 0.95 → crash trigger)
- Entropy loop vectorized → single kernel launch (was B×3 launches → MISSING_GSFRAME crash)
- `label_smoothing_factor=0.1` to prevent loss collapse to ~0 by step 50
- `bf16=False` for 7B (incompatible with 8-bit quant)
- `gradient_checkpointing` enabled for 7B

In [1]:
# Cell 0: Pick model
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:256"

import torch
torch.cuda.set_per_process_memory_fraction(0.85)  # was 0.95 — crash trigger

# TRAIN_MODEL = "qwen25_1p5b_base"
TRAIN_MODEL = "qwen25_3b_base"
# TRAIN_MODEL = "qwen25_7b_base"

print(f"Will train: {TRAIN_MODEL}")
print(f"Max VRAM for PyTorch: {0.85 * 24:.1f} GB")

Will train: qwen25_3b_base
Max VRAM for PyTorch: 20.4 GB


In [2]:
# Cell 1: Imports
import os, sys, json, math, re
from typing import List, Dict, Any, Tuple
from tqdm import tqdm

import torch
from torch.utils.data import Dataset
from transformers import TrainingArguments, Trainer

sys.path.insert(0, os.path.expanduser("~/kd_project"))
from shared_utils import (
    RUNS_DIR, MAX_SEQ_LEN, EPOCHS, LR, SEED, LAMBDA, STUDENTS,
    load_data, load_student,
    compute_alpha, compute_mean_confidence,
    find_decision_span_chars, teacher_section_entropy_mean,
    build_prompt_text, FlexibleCollator,
)

OUT_DIR = os.path.join(RUNS_DIR, "m5_section_routed")
os.makedirs(OUT_DIR, exist_ok=True)

W_CW = 1.0
W_ENT = 0.05

prompts, teacher_rows = load_data()
MEAN_C = compute_mean_confidence(teacher_rows)
print(f"Mean confidence: {MEAN_C:.6f}")

c:\Users\adishalit1\AppData\Local\anaconda3\envs\kd\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Loaded 5000 teacher samples from: data\clinical_pharm_teacher_train_6000.jsonl
Mean confidence: 0.999763


In [3]:
# Cell 2: Precomputed Dataset — decision tokens only
class M5DatasetFast(Dataset):
    def __init__(self, rows, prompts, tokenizer, is_instruct, mean_c):
        print("  Precomputing dataset...")
        self.items = []
        for idx in tqdm(range(len(rows)), desc="  Tokenizing"):
            r = rows[idx]
            prompt = prompts[r["id"]]
            answer = r["teacher_text"]
            prompt_text = build_prompt_text(tokenizer, prompt, is_instruct)
            full_text = prompt_text + answer
            answer_start = len(prompt_text)
            enc = tokenizer(full_text, truncation=True, max_length=MAX_SEQ_LEN, return_offsets_mapping=True)
            input_ids, offsets = enc["input_ids"], enc["offset_mapping"]

            # Labels: ONLY decision tokens
            labels = [-100] * len(input_ids)
            dec_mask = torch.zeros(len(input_ids), dtype=torch.bool)
            dec_span = find_decision_span_chars(answer)
            if dec_span:
                ds, de = dec_span
                full_ds, full_de = answer_start + ds, answer_start + de
                for i, (s, e) in enumerate(offsets):
                    if e <= s: continue
                    if s >= full_ds and s < full_de:
                        labels[i] = input_ids[i]
                        dec_mask[i] = True

            alpha = compute_alpha(r, mean_c)
            ent_teacher = teacher_section_entropy_mean(r, dec_span)

            self.items.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "attention_mask": torch.ones(len(input_ids), dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long),
                "dec_mask": dec_mask,
                "alpha": torch.tensor(alpha, dtype=torch.float),
                "ent_teacher": ent_teacher.float(),
            })
        print(f"  ✅ Precomputed {len(self.items)} samples")

    def __len__(self): return len(self.items)
    def __getitem__(self, idx): return self.items[idx]

print("✅ Dataset ready — decision tokens only")

✅ Dataset ready — decision tokens only


In [4]:
# Cell 3: Trainer — CW-CE + entropy on decision tokens
# FIX: entropy loop with immediate del to avoid OOM on [B,T,V] tensor
class M5Trainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._ce_loss = torch.nn.CrossEntropyLoss(reduction="none")

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        dec_mask = inputs.pop("dec_mask")
        alpha = inputs.pop("alpha")
        ent_teacher = inputs.pop("ent_teacher")

        outputs = model(**inputs)
        logits = outputs.logits
        sl = logits[:, :-1, :].contiguous()
        ll = labels[:, 1:].contiguous()
        dm = dec_mask[:, 1:]
        vocab, B = sl.size(-1), sl.size(0)

        tl = self._ce_loss(sl.view(-1, vocab), ll.view(-1)).view(ll.size())
        active = (ll != -100).float()

        # CW-CE on decision tokens
        dec_active = dm.float() * active
        dec_denom = dec_active.sum(dim=1).clamp(min=1.0)
        per_sample = (tl * dec_active).sum(dim=1) / dec_denom
        L_cw = (per_sample * alpha).mean()

        # Entropy matching — loop but only on decision positions (~1-3 tokens)
        # Each iter holds [num_dec, V] ≈ 0.9MB, not [B, T, V] ≈ 2.4GB
        dec_active_mask = dm & (ll != -100)
        ent_student = torch.zeros(B, device=logits.device)
        for b in range(B):
            idx = dec_active_mask[b].nonzero(as_tuple=True)[0]
            if len(idx) == 0:
                continue
            local_logits = sl[b, idx, :]                                     # [num_dec, V]
            local_probs = torch.softmax(local_logits, dim=-1)                # [num_dec, V]
            local_ent = -(local_probs * torch.log(local_probs + 1e-9)).sum(-1)  # [num_dec]
            ent_student[b] = local_ent.mean()
            del local_logits, local_probs, local_ent                         # free immediately

        L_ent = W_ENT * ((ent_student - ent_teacher.to(logits.device)) ** 2).mean()

        loss = W_CW * L_cw + L_ent
        return (loss, outputs) if return_outputs else loss

print("✅ Trainer ready")

✅ Trainer ready


In [ ]:
# Cell 4: Train
cfg = STUDENTS[TRAIN_MODEL]
print(f"\n{'='*50} M5: {TRAIN_MODEL} {'='*50}")
out_path = os.path.join(OUT_DIR, TRAIN_MODEL)
os.makedirs(out_path, exist_ok=True)

tokenizer, model = load_student(TRAIN_MODEL, cfg)

# 7B: gradient checkpointing to reduce activation memory
if "7b" in TRAIN_MODEL:
    model.gradient_checkpointing_enable()
    print("  ✅ Gradient checkpointing enabled for 7B")

dataset = M5DatasetFast(teacher_rows, prompts, tokenizer, cfg["is_instruct"], MEAN_C)
collator = FlexibleCollator(tokenizer, extra_1d_fields=["dec_mask"],
                            extra_scalar_fields=["alpha", "ent_teacher"])

if "1p5b" in TRAIN_MODEL:
    bs, ga = 4, 8
elif "3b" in TRAIN_MODEL:
    bs, ga = 2, 16
else:  # 7b
    bs, ga = 1, 32

# bf16 incompatible with 8-bit quantization used for 7B
use_bf16 = "7b" not in TRAIN_MODEL

trainer = M5Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=out_path,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=bs,
        gradient_accumulation_steps=ga,
        learning_rate=LR,
        logging_steps=25,
        save_strategy="no",
        bf16=use_bf16,
        seed=SEED,
        report_to="none",
        remove_unused_columns=False,
        dataloader_num_workers=0,
        label_smoothing_factor=0.1,  # prevents loss collapse to ~0 on decision-only supervision
    ),
    train_dataset=dataset,
    data_collator=collator,
)
trainer.train()
model.save_pretrained(out_path)
tokenizer.save_pretrained(out_path)
print(f"\n✅ {TRAIN_MODEL} saved → {out_path}")
print("⚠️ To train next model: change Cell 0, RESTART KERNEL, run all cells")


================================================== M5: qwen25_3b_base ==================================================
  Loading qwen25_3b_base from Qwen/Qwen2.5-3B...


`torch_dtype` is deprecated! Use `dtype` instead!
c:\Users\adishalit1\AppData\Local\anaconda3\envs\kd\Lib\site-packages\accelerate\utils\modeling.py:804: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  _ = torch.tensor([0], device=i)
Loading weights: 100%|██████████| 434/434 [00:04<00:00, 99.94it/s] 


  ✅ qwen25_3b_base: 7,372,800 trainable / 3,093,311,488 total params
  Precomputing dataset...


  Tokenizing: 100%|██████████| 5000/5000 [00:05<00:00, 837.71it/s]


  ✅ Precomputed 5000 samples


Step,Training Loss
25,0.229340
50,0.000096
75,0.000022
100,0.000078
